# Amazon Mattress Price 분석 : 미국법인 ELDP 30 제품

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.ticker import PercentFormatter
import numpy as np
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

In [2]:
import os
import sys

import io
from io import BytesIO
import csv

import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

In [3]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../market-analysis-project-91130-5213911f50a5.json"

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bqclient = bigquery.Client(credentials=credentials, project=project_id,)

In [13]:
# get the data from BigQuery
sql = f"""
select * from wook.amz_zns30_promo_with_comp
where date >= '2026-01-01'
"""

df_source = bqclient.query(sql).to_dataframe()

In [15]:
print(df_source)

      inch  size  brand        date        asin              sku  \
0      6.0  Twin  ZINUS  2026-03-18  B0CSJMCXKB  ZU-MFMA1OZI-06T   
1      6.0  Twin  ZINUS  2026-03-17  B0CSJMCXKB  ZU-MFMA1OZI-06T   
2      6.0  Twin  ZINUS  2026-03-16  B0CSJMCXKB  ZU-MFMA1OZI-06T   
3      6.0  Twin  ZINUS  2026-03-15  B0CSJMCXKB  ZU-MFMA1OZI-06T   
4      6.0  Twin  ZINUS  2026-03-14  B0CSJMCXKB  ZU-MFMA1OZI-06T   
...    ...   ...    ...         ...         ...              ...   
2305   8.0  Twin  ZINUS  2026-01-05  B0CSJTKX7K  ZU-MFMA1OZI-08T   
2306   8.0  Twin  ZINUS  2026-01-04  B0CSJTKX7K  ZU-MFMA1OZI-08T   
2307   8.0  Twin  ZINUS  2026-01-03  B0CSJTKX7K  ZU-MFMA1OZI-08T   
2308   8.0  Twin  ZINUS  2026-01-02  B0CSJTKX7K  ZU-MFMA1OZI-08T   
2309   8.0  Twin  ZINUS  2026-01-01  B0CSJTKX7K  ZU-MFMA1OZI-08T   

                category collection  buybox_price  list_price  ...  \
0     10.FOAM MATTRESSES     MFMBHD         94.46       95.99  ...   
1     10.FOAM MATTRESSES     MFMBHD        

In [17]:
"""
Target SRP 적정성 분석 v5 (최종)
=================================
변경: buybox 기준 판정 + ASP 기준 판정 → 실질 판정(effective_assessment)
핵심: buybox가 OPTIMAL이라도 ASP 이탈이 크면 실질적으로 더 많이 할인 중
"""

import pandas as pd
import numpy as np
from datetime import date

KEYS = ['collection', 'asin', 'size', 'inch']

# 임계값 (조정 가능)
THRESHOLDS = {
    'srp_too_low':          -0.02,   # buybox/asp > SRP
    'optimal_upper':         0.05,   # 0~5%: 아마존이 SRP 수용
    'normal_upper':          0.10,   # 5~10%: 일반적 할인
    'high_upper':            0.15,   # 10~15%: 과도 할인
    # 15%+: EXCESSIVE
    'stability_stable':      0.02,   # gap_pct_std 2% 이하
    'stability_moderate':    0.05,   # 5% 이하
}


# ============================================================
# 자사 가격: 일자별 평균
# ============================================================
def get_own_price_avg(df):
    daily = df.groupby([*KEYS, 'date']).agg(
        target_srp=('target_srp', 'first'),
        buybox_price=('buybox_price', 'first'),
        sales_rank=('sales_rank', 'first'),
        shipped_units=('shipped_units', 'first'),
        shipped_revenue=('shipped_revenue', 'first'),
        net_ppm=('net_ppm', 'first'),
    ).reset_index()

    avg = daily.groupby(KEYS).agg(
        target_srp=('target_srp', 'mean'),
        buybox_price=('buybox_price', 'mean'),
        sales_rank=('sales_rank', 'mean'),
        shipped_units=('shipped_units', 'sum'),
        shipped_revenue=('shipped_revenue', 'sum'),
        net_ppm=('net_ppm', 'mean'),
        date_count=('date', 'nunique'),
    ).reset_index()

    for col in ['target_srp', 'buybox_price', 'net_ppm']:
        avg[col] = avg[col].round(2)
    avg['sales_rank'] = avg['sales_rank'].round(0)

    return avg


# ============================================================
# 일자별 변동성 분석
# ============================================================
def get_buybox_volatility(df):
    daily = df.groupby([*KEYS, 'date']).agg(
        buybox_price=('buybox_price', 'first'),
        target_srp=('target_srp', 'first'),
    ).reset_index()

    daily['daily_gap_pct'] = (
        (daily['target_srp'] - daily['buybox_price']) / daily['target_srp']
    )

    vol = daily.groupby(KEYS).agg(
        buybox_min=('buybox_price', 'min'),
        buybox_max=('buybox_price', 'max'),
        buybox_std=('buybox_price', 'std'),
        gap_pct_min=('daily_gap_pct', 'min'),
        gap_pct_max=('daily_gap_pct', 'max'),
        gap_pct_std=('daily_gap_pct', 'std'),
        days_buybox_above_srp=('daily_gap_pct', lambda x: (x < 0).sum()),
        days_buybox_below_srp=('daily_gap_pct', lambda x: (x > 0).sum()),
        total_days=('daily_gap_pct', 'count'),
    ).reset_index()

    vol['buybox_range'] = (vol['buybox_max'] - vol['buybox_min']).round(2)
    vol['buybox_std'] = vol['buybox_std'].round(2)
    vol['gap_pct_std'] = vol['gap_pct_std'].round(4)
    vol['pct_days_above_srp'] = (vol['days_buybox_above_srp'] / vol['total_days']).round(4)

    return vol


# ============================================================
# 판정 함수 (buybox, ASP 공통)
# ============================================================
def assign_assessment(pct_series):
    """이탈률 기준 5단계 판정 (buybox, ASP 공통 사용)"""
    T = THRESHOLDS
    conditions = [
        pct_series < T['srp_too_low'],
        (pct_series >= T['srp_too_low']) & (pct_series <= T['optimal_upper']),
        (pct_series > T['optimal_upper']) & (pct_series <= T['normal_upper']),
        (pct_series > T['normal_upper']) & (pct_series <= T['high_upper']),
        pct_series > T['high_upper'],
    ]
    choices = [
        'SRP_TOO_LOW',
        'OPTIMAL',
        'NORMAL_DISCOUNT',
        'HIGH_DISCOUNT',
        'EXCESSIVE_DISCOUNT',
    ]
    return np.select(conditions, choices, default='CHECK')


def assign_stability(gap_pct_std_series):
    """buybox 안정성 판정"""
    T = THRESHOLDS
    conditions = [
        gap_pct_std_series <= T['stability_stable'],
        gap_pct_std_series <= T['stability_moderate'],
        gap_pct_std_series > T['stability_moderate'],
    ]
    return np.select(conditions, ['STABLE', 'MODERATE', 'VOLATILE'], default='CHECK')


# ============================================================
# 실질 판정 (buybox + ASP 결합)
# ============================================================
def assign_effective_assessment(row):
    """
    buybox 판정과 ASP 판정 중 더 나쁜 쪽을 실질 판정으로 채택
    단, buybox와 ASP 판정이 다르면 그 차이도 기록
    """
    rank = {
        'EXCESSIVE_DISCOUNT': 0,
        'HIGH_DISCOUNT': 1,
        'NORMAL_DISCOUNT': 2,
        'SRP_TOO_LOW': 3,
        'OPTIMAL': 4,
    }
    bb = row['buybox_assessment']
    asp = row['asp_assessment']

    bb_rank = rank.get(bb, 5)
    asp_rank = rank.get(asp, 5)

    # 더 나쁜(rank 낮은) 판정 채택
    if asp_rank < bb_rank:
        return asp   # ASP가 더 나쁨 → ASP 기준 채택
    else:
        return bb    # buybox와 같거나 buybox가 더 나쁨


# ============================================================
# 핵심 분석
# ============================================================
def evaluate_srp(df):
    """메인 분석"""

    own_avg = get_own_price_avg(df)
    vol = get_buybox_volatility(df)

    result = own_avg.merge(vol, on=KEYS, how='left')

    # ── buybox 기준 지표 ──
    result['srp_buybox_gap'] = (result['target_srp'] - result['buybox_price']).round(2)
    result['srp_buybox_pct'] = (
        result['srp_buybox_gap'] / result['target_srp']
    ).round(4)

    # ── ASP 기준 지표 ──
    result['asp'] = (result['shipped_revenue'] / result['shipped_units']).round(2)
    result['srp_asp_gap'] = (result['target_srp'] - result['asp']).round(2)
    result['srp_asp_pct'] = (
        result['srp_asp_gap'] / result['target_srp']
    ).round(4)

    # ── buybox vs ASP 차이 (추가 할인 규모) ──
    result['buybox_asp_gap'] = (result['buybox_price'] - result['asp']).round(2)
    result['buybox_asp_pct'] = (
        result['buybox_asp_gap'] / result['buybox_price']
    ).round(4)

    # ── 3개 판정 ──
    result['buybox_assessment'] = assign_assessment(result['srp_buybox_pct'])
    result['asp_assessment'] = assign_assessment(result['srp_asp_pct'])
    result['buybox_stability'] = assign_stability(result['gap_pct_std'])

    # 실질 판정: buybox와 ASP 중 더 나쁜 쪽 채택
    result['effective_assessment'] = result.apply(assign_effective_assessment, axis=1)

    # buybox와 ASP 판정 불일치 플래그
    result['assessment_mismatch'] = result['buybox_assessment'] != result['asp_assessment']

    # ── 정렬 ──
    priority = {
        'EXCESSIVE_DISCOUNT': 0, 'HIGH_DISCOUNT': 1,
        'SRP_TOO_LOW': 2, 'NORMAL_DISCOUNT': 3,
        'OPTIMAL': 4, 'CHECK': 5
    }
    result['_sort'] = result['effective_assessment'].map(priority)
    result = result.sort_values(['_sort', 'srp_asp_pct'], ascending=[True, False])
    result = result.drop(columns='_sort')

    return result


# ============================================================
# Collection 서머리
# ============================================================
def collection_summary(result):
    summary = result.groupby('collection').agg(
        asin_count=('asin', 'nunique'),
        optimal_count=('effective_assessment', lambda x: (x == 'OPTIMAL').sum()),
        normal_count=('effective_assessment', lambda x: (x == 'NORMAL_DISCOUNT').sum()),
        high_count=('effective_assessment', lambda x: (x == 'HIGH_DISCOUNT').sum()),
        excessive_count=('effective_assessment', lambda x: (x == 'EXCESSIVE_DISCOUNT').sum()),
        avg_target_srp=('target_srp', 'mean'),
        avg_buybox=('buybox_price', 'mean'),
        avg_asp=('asp', 'mean'),
        avg_srp_buybox_pct=('srp_buybox_pct', 'mean'),
        avg_srp_asp_pct=('srp_asp_pct', 'mean'),
        avg_buybox_asp_pct=('buybox_asp_pct', 'mean'),
        avg_net_ppm=('net_ppm', 'mean'),
        mismatch_count=('assessment_mismatch', 'sum'),
    ).reset_index()

    summary['healthy_rate'] = (
        (summary['optimal_count'] + summary['normal_count']) / summary['asin_count']
    ).round(2)

    return summary.sort_values('healthy_rate')


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    df = df_source.copy()
    
    result = evaluate_srp(df)
    summary = collection_summary(result)

    display_cols = [
        'collection', 'asin', 'size', 'inch',
        'target_srp', 'buybox_price', 'asp',
        'srp_buybox_pct', 'srp_asp_pct', 'buybox_asp_pct',
        'buybox_assessment', 'asp_assessment', 'effective_assessment',
        'assessment_mismatch',
        'buybox_stability', 'buybox_range', 'gap_pct_std',
        'net_ppm', 'sales_rank',
    ]

    print("=== ASIN별 SRP 적정성 (최종) ===")
    print(result[display_cols].to_string(index=False))
    print("\n=== Collection 서머리 ===")
    print(summary.to_string(index=False))


=== ASIN별 SRP 적정성 (최종) ===
collection       asin        size  inch  target_srp  buybox_price    asp  srp_buybox_pct  srp_asp_pct  buybox_asp_pct buybox_assessment     asp_assessment effective_assessment  assessment_mismatch buybox_stability  buybox_range  gap_pct_std  net_ppm  sales_rank
       FMS B0CSJK3863 Short Queen   8.0      199.99        176.51 161.07          0.1174       0.1946          0.0875     HIGH_DISCOUNT EXCESSIVE_DISCOUNT   EXCESSIVE_DISCOUNT                 True         VOLATILE        108.08       0.1174    51.63       203.0
    MFMAMF B0CSK2VDKG       Queen  12.0      249.99        217.29 201.95          0.1308       0.1922          0.0706     HIGH_DISCOUNT EXCESSIVE_DISCOUNT   EXCESSIVE_DISCOUNT                 True         VOLATILE         91.27       0.0988    60.48        36.0
  OLB GTFM B0CKYZPPMJ        Twin   8.0      119.99        107.18  99.93          0.1068       0.1672          0.0676     HIGH_DISCOUNT EXCESSIVE_DISCOUNT   EXCESSIVE_DISCOUNT            

In [21]:
"""
프로모션 펀딩 적정성 분석 v2
=============================
수정: funding rate 분모를 ASP → target_srp로 변경
이유: ASP는 프로모션 결과값이므로 프로모션 전 기준가인 target_srp가 적합
"""

import pandas as pd
import numpy as np
from datetime import date

KEYS = ['collection', 'asin', 'size', 'inch']

# Funding Rate 판정 임계값 (target_srp 기준)
FUNDING_THRESHOLDS = {
    'low':       0.02,   # 2% 미만: 과소
    'adequate':  0.10,   # 2~10%: 적정
    'high':      0.20,   # 10~20%: 과다
    # 20%+: 과잉
}


# ============================================================
# 데이터 로드
# ============================================================
def load_promo_data(promo_path):
    promo = pd.read_csv(promo_path)
    promo.columns = promo.columns.str.strip()
    promo['Per Unit $'] = promo['Per Unit $'].astype(float)
    promo['funding amt'] = promo['funding amt'].astype(float)
    promo['ASIN'] = promo['ASIN'].str.strip()
    return promo


def load_market_data(ms_path):
    ms = pd.read_csv(ms_path)
    ms.columns = ms.columns.str.strip()
    ms['inch'] = ms['inch'].astype(str).str.strip('"').str.strip()
    ms['inch_num'] = pd.to_numeric(ms['inch'], errors='coerce')
    ms = ms[ms['inch_num'].notna()].copy()
    return ms


# ============================================================
# SRP 분석 결과 (v5에서 가져온 데이터)
# ============================================================
def get_srp_result():
    """v5 분석 결과 — 실제 환경에서는 evaluate_srp() 결과를 직접 사용"""
    return pd.DataFrame({
        'asin': ['B0CKZ1CK1H','B0CKYZC93L','B0CKYZ3B83','B0CKYZCVXK','B0CKZ1RXKH',
                 'B0CSJTKX7K','B0CSK2VDKG','B0CP1LR1PW','B0CKYXPC4Z','B0CKYYHD47',
                 'B0CKYZ4DJB','B0CKYY27YP','B0CSJTBM1L','B0CKYZPPMJ','B0CKYYL8B9',
                 'B0CKYZHV5D','B0CSJQGDC6','B0CSK5C8MJ','B0CKYYYYYF','B0CKYRS739',
                 'B0CSJTJCCS','B0CSJTDWXS','B0CKYZJHQ6','B0CKYZ6K4V','B0CSJK3863',
                 'B0CSJMCXKB','B0CKYSMM1J','B0CKYYZMY3','B0CKYYCLRW','B0CKYZRRCX'],
        'collection': ['OLB GTFM','OLB GTFM','OLB GTFM','OLB GTFM','OLB GTFM',
                       'MFMBHD','MFMAMF','OLB GTFM','OLB FGM','OLB FGM',
                       'OLB FGM','OLB GTFM','MFMBHD','OLB GTFM','OLB GTFM',
                       'OLB GTFM','MFMAMF','MFMBHD','OLB FGM','OLB BNSM',
                       'MFMBHD','MFMBHD','OLB GTFM','OLB FGM','FMS',
                       'MFMBHD','OLB BNSM','OLB GTFM','OLB GTFM','OLB GTF'],
        'size': ['King','Queen','Queen','Twin','Full',
                 'Twin','Queen','Twin','King','Queen',
                 'Queen','Full','Queen','Twin','Full',
                 'Full','Queen','Queen','Full','Twin',
                 'Twin','Full','Queen','Twin','Short Queen',
                 'Twin','Twin','Twin','Queen','King'],
        'inch': [12.0,12.0,10.0,6.0,12.0,
                 8.0,12.0,5.0,12.0,12.0,
                 10.0,8.0,10.0,8.0,10.0,
                 6.0,10.0,12.0,10.0,6.0,
                 5.0,8.0,6.0,8.0,8.0,
                 6.0,8.0,10.0,8.0,10.0],
        'target_srp': [359.99,299.99,219.99,99.99,239.99,
                       99.99,249.99,79.99,359.99,299.99,
                       219.99,139.99,179.99,119.99,209.99,
                       127.99,179.99,249.99,209.99,89.99,
                       79.99,129.99,129.99,119.99,199.99,
                       79.99,115.99,179.99,159.99,349.99],
        'buybox_price': [350.46,282.25,215.88,93.80,234.47,
                         98.54,217.29,73.81,322.76,283.56,
                         219.54,135.20,178.81,107.18,203.21,
                         117.80,172.53,226.45,210.67,88.09,
                         76.73,127.09,129.56,105.07,176.51,
                         79.85,110.88,163.75,163.87,313.28],
        'asp': [338.53,273.52,214.49,88.89,229.14,
                97.55,201.95,71.31,317.14,279.78,
                219.29,134.65,178.44,99.93,198.74,
                113.03,163.07,216.18,210.03,86.24,
                74.59,125.61,127.53,102.15,161.07,
                78.60,104.54,161.92,160.32,312.00],
        'net_ppm': [53.42,55.43,56.07,57.11,52.79,
                    51.66,60.48,55.75,52.26,55.59,
                    49.77,43.08,55.26,55.88,57.28,
                    50.16,60.44,61.88,53.44,53.09,
                    56.02,54.34,46.07,49.22,51.63,
                    52.73,57.99,41.45,53.74,49.20],
        'effective_assessment': [
            'NORMAL_DISCOUNT','NORMAL_DISCOUNT','OPTIMAL','HIGH_DISCOUNT','OPTIMAL',
            'OPTIMAL','EXCESSIVE_DISCOUNT','HIGH_DISCOUNT','HIGH_DISCOUNT','NORMAL_DISCOUNT',
            'OPTIMAL','OPTIMAL','OPTIMAL','EXCESSIVE_DISCOUNT','NORMAL_DISCOUNT',
            'HIGH_DISCOUNT','NORMAL_DISCOUNT','HIGH_DISCOUNT','OPTIMAL','OPTIMAL',
            'NORMAL_DISCOUNT','OPTIMAL','OPTIMAL','HIGH_DISCOUNT','EXCESSIVE_DISCOUNT',
            'OPTIMAL','NORMAL_DISCOUNT','HIGH_DISCOUNT','SRP_TOO_LOW','HIGH_DISCOUNT'],
        'sales_rank': [3,3,3,3,3,6,36,3,24,24,24,3,6,3,3,3,36,6,23,19,6,6,3,23,203,6,19,3,3,3],
    })


# ============================================================
# 핵심 분석
# ============================================================
def analyze_funding(srp_result, promo, market):
    """프로모션 펀딩 적정성 분석"""

    # 병합: SRP 결과 + 프로모션 펀딩
    merged = srp_result.merge(promo, left_on='asin', right_on='ASIN', how='left')

    # 시장 크기 매핑
    merged['inch_str'] = merged['inch'].astype(int).astype(str)
    merged['size_for_ms'] = merged['size'].replace({'Short Queen': 'Queen'})

    merged = merged.merge(
        market[['inch', 'size', 'sales', 'units']].rename(
            columns={'sales': 'market_sales', 'units': 'market_units'}
        ),
        left_on=['inch_str', 'size_for_ms'],
        right_on=['inch', 'size'],
        how='left', suffixes=('', '_ms')
    )

    # ── 핵심 지표 (target_srp 기준) ──

    merged['funding_per_unit'] = merged['Per Unit $']
    merged['funding_total'] = merged['funding amt']

    # implied funded units
    merged['funded_units'] = (merged['funding_total'] / merged['funding_per_unit']).round(0)

    # ★ Funding Rate: target_srp 기준 (프로모션 전 기준가)
    merged['funding_rate'] = (merged['funding_per_unit'] / merged['target_srp']).round(4)

    # 참고: ASP 기준 rate (결과값 기반, 비교용)
    merged['funding_rate_vs_asp'] = (merged['funding_per_unit'] / merged['asp']).round(4)

    # 시장 점유율
    merged['unit_share_pct'] = (merged['funded_units'] / merged['market_units'] * 100).round(4)

    # Effective ASP (프로모션 비용 차감 후 실질 수취)
    merged['effective_asp'] = (merged['asp'] - merged['funding_per_unit']).round(2)
    merged['effective_asp_vs_srp_pct'] = (
        (merged['target_srp'] - merged['effective_asp']) / merged['target_srp']
    ).round(4)

    # Funding ROI: 시장 점유율 1% 확보당 펀딩 비용
    merged['funding_per_1pct_share'] = np.where(
        merged['unit_share_pct'] > 0,
        (merged['funding_total'] / merged['unit_share_pct']).round(2),
        np.nan
    )

    # ── Funding 적정성 판정 (target_srp 기준) ──
    T = FUNDING_THRESHOLDS
    conditions = [
        merged['funding_rate'] < T['low'],
        (merged['funding_rate'] >= T['low']) & (merged['funding_rate'] <= T['adequate']),
        (merged['funding_rate'] > T['adequate']) & (merged['funding_rate'] <= T['high']),
        merged['funding_rate'] > T['high'],
    ]
    choices = ['LOW_FUNDING', 'ADEQUATE', 'HIGH_FUNDING', 'EXCESSIVE_FUNDING']
    merged['funding_assessment'] = np.select(conditions, choices, default='CHECK')

    # SRP 판정 × Funding 판정 교차
    merged['srp_funding_combo'] = merged['effective_assessment'] + ' / ' + merged['funding_assessment']

    # 정렬
    merged = merged.sort_values('funding_rate', ascending=False)

    return merged


# ============================================================
# 서머리
# ============================================================
def funding_summary(result):
    """Funding 판정별 서머리"""
    return result.groupby('funding_assessment').agg(
        count=('asin', 'count'),
        avg_funding_rate=('funding_rate', 'mean'),
        avg_per_unit=('funding_per_unit', 'mean'),
        total_funding=('funding_total', 'sum'),
        avg_unit_share=('unit_share_pct', 'mean'),
        avg_net_ppm=('net_ppm', 'mean'),
    ).reset_index()


def collection_funding_summary(result):
    """Collection별 서머리"""
    coll = result.groupby('collection').agg(
        asin_count=('asin', 'nunique'),
        total_funding=('funding_total', 'sum'),
        avg_funding_rate=('funding_rate', 'mean'),
        avg_unit_share=('unit_share_pct', 'mean'),
        avg_net_ppm=('net_ppm', 'mean'),
        funding_adequate=('funding_assessment', lambda x: (x == 'ADEQUATE').sum()),
    ).reset_index()
    coll['adequate_rate'] = (coll['funding_adequate'] / coll['asin_count']).round(2)
    return coll.sort_values('avg_funding_rate', ascending=False)


def market_segment_summary(result):
    """시장 세그먼트별 펀딩 배분"""
    seg = result.groupby(['size_for_ms', 'inch_str']).agg(
        asin_count=('asin', 'count'),
        market_sales=('market_sales', 'first'),
        market_units=('market_units', 'first'),
        total_funding=('funding_total', 'sum'),
        total_funded_units=('funded_units', 'sum'),
        avg_funding_rate=('funding_rate', 'mean'),
    ).reset_index()
    seg['funding_intensity'] = (seg['total_funding'] / seg['market_units'] * 1000).round(2)
    return seg.sort_values('market_sales', ascending=False)


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    promo = load_promo_data('./amz_promo_fund_0322.csv')
    market = load_market_data('./amz_ms_inch_size_0322.csv')
    srp_result = get_srp_result()

    result = analyze_funding(srp_result, promo, market)

    pd.set_option('display.max_columns', 30)
    pd.set_option('display.width', 300)

    # ASIN별 상세
    display_cols = [
        'collection', 'asin', 'size', 'inch',
        'target_srp', 'buybox_price', 'asp',
        'funding_per_unit', 'funding_rate', 'funding_rate_vs_asp',
        'funding_total', 'funded_units',
        'market_units', 'unit_share_pct',
        'effective_asp', 'effective_asp_vs_srp_pct',
        'funding_assessment', 'effective_assessment',
        'net_ppm', 'sales_rank',
    ]
    print("=== ASIN별 프로모션 펀딩 분석 (target_srp 기준) ===")
    print(result[display_cols].to_string(index=False))

    # Funding 판정별 서머리
    print("\n=== Funding 판정별 서머리 ===")
    print(funding_summary(result).to_string(index=False))

    # SRP × Funding 교차표
    print("\n=== SRP 판정 × Funding 판정 교차표 ===")
    print(pd.crosstab(result['effective_assessment'], result['funding_assessment'], margins=True))

    # Collection 서머리
    print("\n=== Collection 서머리 ===")
    print(collection_funding_summary(result).to_string(index=False))

    # 시장 크기 대비
    print("\n=== 시장 크기 vs 펀딩 배분 ===")
    print(market_segment_summary(result).to_string(index=False))

=== ASIN별 프로모션 펀딩 분석 (target_srp 기준) ===
collection       asin        size  inch  target_srp  buybox_price    asp  funding_per_unit  funding_rate  funding_rate_vs_asp  funding_total  funded_units  market_units  unit_share_pct  effective_asp  effective_asp_vs_srp_pct funding_assessment effective_assessment  net_ppm  sales_rank
  OLB GTFM B0CKYYCLRW       Queen   8.0      159.99        163.87 160.32             63.46        0.3966               0.3958          141.0           2.0         26050          0.0077          96.86                    0.3946  EXCESSIVE_FUNDING          SRP_TOO_LOW    53.74           3
  OLB GTFM B0CKYY27YP        Full   8.0      139.99        135.20 134.65             41.98        0.2999               0.3118          224.0           5.0         92428          0.0054          92.67                    0.3380  EXCESSIVE_FUNDING              OPTIMAL    43.08           3
  OLB GTFM B0CKYZJHQ6       Queen   6.0      129.99        129.56 127.53             31.00        

In [42]:
"""
Target SRP 적정성 분석 v4
=========================
경쟁사 데이터 없이 target_srp vs buybox_price만으로 판정
핵심: 아마존의 buybox 행동 자체가 SRP 적정성의 시그널
"""

import pandas as pd
import numpy as np
from datetime import date

KEYS = ['collection', 'asin', 'size', 'inch']


# ============================================================
# 자사 가격: 일자별 평균
# ============================================================
def get_own_price_avg(df):
    daily = df.groupby([*KEYS, 'date']).agg(
        target_srp=('target_srp', 'first'),
        buybox_price=('buybox_price', 'first'),
        sales_rank=('sales_rank', 'first'),
        shipped_units=('shipped_units', 'first'),
        shipped_revenue=('shipped_revenue', 'first'),
        net_ppm=('net_ppm', 'first'),
    ).reset_index()

    avg = daily.groupby(KEYS).agg(
        target_srp=('target_srp', 'mean'),
        buybox_price=('buybox_price', 'mean'),
        sales_rank=('sales_rank', 'mean'),
        shipped_units=('shipped_units', 'sum'),
        shipped_revenue=('shipped_revenue', 'sum'),
        net_ppm=('net_ppm', 'mean'),
        date_count=('date', 'nunique'),
    ).reset_index()

    for col in ['target_srp', 'buybox_price', 'net_ppm']:
        avg[col] = avg[col].round(2)
    avg['sales_rank'] = avg['sales_rank'].round(0)

    return avg


# ============================================================
# 일자별 변동성 분석
# ============================================================
def get_buybox_volatility(df):
    """buybox의 일자별 변동 패턴 분석"""
    daily = df.groupby([*KEYS, 'date']).agg(
        buybox_price=('buybox_price', 'first'),
        target_srp=('target_srp', 'first'),
    ).reset_index()

    # 일자별 SRP 대비 buybox 비율
    daily['daily_gap_pct'] = (
        (daily['target_srp'] - daily['buybox_price']) / daily['target_srp']
    )

    vol = daily.groupby(KEYS).agg(
        buybox_min=('buybox_price', 'min'),
        buybox_max=('buybox_price', 'max'),
        buybox_std=('buybox_price', 'std'),
        gap_pct_min=('daily_gap_pct', 'min'),    # buybox가 SRP 대비 가장 높았던 날
        gap_pct_max=('daily_gap_pct', 'max'),    # buybox가 SRP 대비 가장 낮았던 날
        gap_pct_std=('daily_gap_pct', 'std'),    # 이탈률 변동성
        days_buybox_above_srp=('daily_gap_pct', lambda x: (x < 0).sum()),
        days_buybox_below_srp=('daily_gap_pct', lambda x: (x > 0).sum()),
        total_days=('daily_gap_pct', 'count'),
    ).reset_index()

    vol['buybox_range'] = (vol['buybox_max'] - vol['buybox_min']).round(2)
    vol['buybox_std'] = vol['buybox_std'].round(2)
    vol['gap_pct_std'] = vol['gap_pct_std'].round(4)
    vol['pct_days_above_srp'] = (vol['days_buybox_above_srp'] / vol['total_days']).round(4)

    return vol


# ============================================================
# 핵심 분석: SRP vs Buybox 판정
# ============================================================
def evaluate_srp(df):
    """메인 분석"""

    own_avg = get_own_price_avg(df)
    vol = get_buybox_volatility(df)

    result = own_avg.merge(vol, on=KEYS, how='left')

    # ── 핵심 지표: SRP vs Buybox 괴리 ──
    result['srp_buybox_gap'] = (result['target_srp'] - result['buybox_price']).round(2)
    result['srp_buybox_pct'] = (
        result['srp_buybox_gap'] / result['target_srp']
    ).round(4)

    # ── ASP (실질 판매 단가) ──
    result['asp'] = (result['shipped_revenue'] / result['shipped_units']).round(2)
    result['asp_vs_srp_pct'] = (
        (result['target_srp'] - result['asp']) / result['target_srp']
    ).round(4)

    # ── 판정 로직 ──
    # buybox > SRP: 아마존이 SRP보다 비싸게 판매 → SRP가 낮을 수 있음
    # buybox ≈ SRP (0~5%): 적정. 아마존이 SRP를 수용
    # buybox < SRP (5~10%): 정상 할인 범위
    # buybox << SRP (10~15%): 과도한 이탈, 주의
    # buybox <<< SRP (15%+): 심각한 괴리, SRP 재검토

    conditions = [
        # (1) buybox > SRP: SRP 인상 검토
        result['srp_buybox_pct'] < -0.02,

        # (2) buybox ≈ SRP (0~5%): 아마존이 할인 없이 SRP 수용
        (result['srp_buybox_pct'] >= -0.02) & (result['srp_buybox_pct'] <= 0.05),

        # (3) 정상 할인 (5~10%): 일반적인 아마존 할인 운영
        (result['srp_buybox_pct'] > 0.05) & (result['srp_buybox_pct'] <= 0.10),

        # (4) 과도 이탈 (10~15%): 아마존이 적극적으로 가격 인하 중
        (result['srp_buybox_pct'] > 0.10) & (result['srp_buybox_pct'] <= 0.15),

        # (5) 심각 괴리 (15%+): SRP가 시장과 괴리
        result['srp_buybox_pct'] > 0.15,
    ]

    choices = [
        'SRP_TOO_LOW',           # buybox > SRP → SRP 인상 여력
        'OPTIMAL',               # buybox ≈ SRP → 가장 이상적
        'NORMAL_DISCOUNT',       # 5~10% 할인 → 정상 운영
        'HIGH_DISCOUNT',         # 10~15% 할인 → 주의, 모니터링
        'EXCESSIVE_DISCOUNT',    # 15%+ 할인 → SRP 재검토 필요
    ]

    result['srp_assessment'] = np.select(conditions, choices, default='CHECK')

    # ── 추가 시그널: buybox 안정성 판정 ──
    conditions_stability = [
        result['gap_pct_std'] <= 0.02,       # 이탈률 표준편차 2% 이내
        result['gap_pct_std'] <= 0.05,       # 5% 이내
        result['gap_pct_std'] > 0.05,        # 5% 초과
    ]
    choices_stability = ['STABLE', 'MODERATE', 'VOLATILE']
    result['buybox_stability'] = np.select(conditions_stability, choices_stability, default='CHECK')

    # ── 종합 신호 ──
    # OPTIMAL + STABLE = 가장 건강
    # OPTIMAL + VOLATILE = SRP는 맞지만 buybox가 불안정
    # HIGH/EXCESSIVE + STABLE = 아마존이 일관되게 할인 → SRP 인하 신호
    # SRP_TOO_LOW + STABLE = 일관되게 SRP 초과 → SRP 인상 검토

    # 정렬
    priority = {
        'EXCESSIVE_DISCOUNT': 0, 'HIGH_DISCOUNT': 1,
        'SRP_TOO_LOW': 2, 'NORMAL_DISCOUNT': 3,
        'OPTIMAL': 4, 'CHECK': 5
    }
    result['_sort'] = result['srp_assessment'].map(priority)
    result = result.sort_values(['_sort', 'srp_buybox_pct'], ascending=[True, False])
    result = result.drop(columns='_sort')

    return result


# ============================================================
# Collection 서머리
# ============================================================
def collection_summary(result):
    summary = result.groupby('collection').agg(
        asin_count=('asin', 'nunique'),
        optimal_count=('srp_assessment', lambda x: (x == 'OPTIMAL').sum()),
        normal_count=('srp_assessment', lambda x: (x == 'NORMAL_DISCOUNT').sum()),
        avg_target_srp=('target_srp', 'mean'),
        avg_buybox=('buybox_price', 'mean'),
        avg_srp_buybox_pct=('srp_buybox_pct', 'mean'),
        avg_net_ppm=('net_ppm', 'mean'),
        avg_sales_rank=('sales_rank', 'mean'),
    ).reset_index()

    summary['healthy_rate'] = (
        (summary['optimal_count'] + summary['normal_count']) / summary['asin_count']
    ).round(2)

    return summary.sort_values('healthy_rate')


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    # df = df_source[df_source['date'] >= date(2026, 1, 1)]
    df = df_source[pd.to_datetime(df_source['date']) >= '2026-01-01']
    
    result = evaluate_srp(df)
    summary = collection_summary(result)

    display_cols = [
        'collection', 'asin', 'size', 'inch',
        'target_srp', 'buybox_price', 'srp_buybox_pct',
        'srp_assessment', 'buybox_stability',
        'buybox_min', 'buybox_max', 'buybox_range', 'gap_pct_std',
        'pct_days_above_srp',
        'asp', 'asp_vs_srp_pct',
        'net_ppm', 'sales_rank',
    ]

    print("=== ASIN별 SRP 적정성 ===")
    print(result[display_cols].to_string(index=False))
    print("\n=== Collection 서머리 ===")
    print(summary.to_string(index=False))


=== ASIN별 SRP 적정성 ===
collection       asin        size  inch  target_srp  buybox_price  srp_buybox_pct  srp_assessment buybox_stability  buybox_min  buybox_max  buybox_range  gap_pct_std  pct_days_above_srp    asp  asp_vs_srp_pct  net_ppm  sales_rank
    MFMAMF B0CSK2VDKG       Queen  12.0      249.99        217.29          0.1308   HIGH_DISCOUNT         VOLATILE      190.00      281.27         91.27       0.0988              0.0263 201.95          0.1922    60.48        36.0
   OLB FGM B0CKYZ6K4V        Twin   8.0      119.99        105.07          0.1243   HIGH_DISCOUNT         VOLATILE       84.50      127.99         43.49       0.0888              0.0133 102.15          0.1487    49.22        23.0
       FMS B0CSJK3863 Short Queen   8.0      199.99        176.51          0.1174   HIGH_DISCOUNT         VOLATILE      137.30      245.38        108.08       0.1174              0.0526 161.07          0.1946    51.63       203.0
  OLB GTFM B0CKYZPPMJ        Twin   8.0      119.99       

In [40]:
"""
Target SRP 적정성 분석 v3
=========================
list_price 제외 (아마존 NetPPM용 → 통제 불가)
target_srp vs buybox_price 2축 분석

축1: target_srp가 시장에서 적정한가? (경쟁사 대비)
축2: buybox가 target_srp를 잘 따르는가? (가격 통제력)
"""

import pandas as pd
import numpy as np
from datetime import date

KEYS = ['collection', 'asin', 'size', 'inch']

# 임계값 (조정 가능)
DRIFT_THRESHOLD = 0.10      # buybox 이탈 기준: SRP 대비 10%
COMP_HIGH_THRESHOLD = 0.15  # 경쟁사 대비 고가 기준: +15%
COMP_LOW_THRESHOLD = -0.15  # 경쟁사 대비 저가 기준: -15%


# ============================================================
# 자사 가격: 일자별 평균
# ============================================================
def get_own_price_avg(df):
    daily = df.groupby([*KEYS, 'date']).agg(
        target_srp=('target_srp', 'first'),
        buybox_price=('buybox_price', 'first'),
        sales_rank=('sales_rank', 'first'),
        shipped_units=('shipped_units', 'first'),
        shipped_revenue=('shipped_revenue', 'first'),
        net_ppm=('net_ppm', 'first'),
    ).reset_index()

    avg = daily.groupby(KEYS).agg(
        target_srp=('target_srp', 'mean'),
        buybox_price=('buybox_price', 'mean'),
        sales_rank=('sales_rank', 'mean'),
        shipped_units=('shipped_units', 'sum'),
        shipped_revenue=('shipped_revenue', 'sum'),
        net_ppm=('net_ppm', 'mean'),
        date_count=('date', 'nunique'),
    ).reset_index()

    avg['target_srp'] = avg['target_srp'].round(2)
    avg['buybox_price'] = avg['buybox_price'].round(2)
    avg['net_ppm'] = avg['net_ppm'].round(2)

    return avg


# ============================================================
# 경쟁사 가격: 경쟁사별 일자평균 → 통계
# ============================================================
def get_comp_stats(df):
    # Step 1: 일자별
    comp_daily = df.groupby([*KEYS, 'comp_asin', 'comp_brand', 'date']).agg(
        comp_buybox_price=('comp_buybox_price', 'first'),
        comp_sales_rank=('comp_sales_rank', 'first'),
    ).reset_index()

    # Step 2: 경쟁사별 일자 평균
    comp_avg = comp_daily.groupby([*KEYS, 'comp_asin', 'comp_brand']).agg(
        comp_buybox_avg=('comp_buybox_price', 'mean'),
        comp_rank_avg=('comp_sales_rank', 'mean'),
    ).reset_index()
    comp_avg['comp_buybox_avg'] = comp_avg['comp_buybox_avg'].round(2)

    # Step 3: ASIN별 통계
    stats = comp_avg.groupby(KEYS).agg(
        comp_buybox_median=('comp_buybox_avg', 'median'),
        comp_buybox_mean=('comp_buybox_avg', 'mean'),
        comp_buybox_p25=('comp_buybox_avg', lambda x: x.quantile(0.25)),
        comp_buybox_p75=('comp_buybox_avg', lambda x: x.quantile(0.75)),
        comp_buybox_min=('comp_buybox_avg', 'min'),
        comp_buybox_max=('comp_buybox_avg', 'max'),
        comp_count=('comp_asin', 'nunique'),
    ).reset_index()

    for col in ['comp_buybox_median', 'comp_buybox_mean', 'comp_buybox_p25',
                'comp_buybox_p75', 'comp_buybox_min', 'comp_buybox_max']:
        stats[col] = stats[col].round(2)

    # BSR 가중평균
    def bsr_wavg(group):
        valid = group.dropna(subset=['comp_buybox_avg', 'comp_rank_avg'])
        valid = valid[valid['comp_rank_avg'] > 0]
        if len(valid) == 0:
            return np.nan
        w = 1.0 / valid['comp_rank_avg'].clip(lower=1)
        return round(np.average(valid['comp_buybox_avg'], weights=w), 2)

    bsr = comp_avg.groupby(KEYS).apply(bsr_wavg).reset_index(name='comp_buybox_bsr_wavg')
    stats = stats.merge(bsr, on=KEYS, how='left')

    return comp_avg, stats


# ============================================================
# 핵심 분석: 2축 판정
# ============================================================
def evaluate_srp(df):
    """메인 분석 함수"""

    own_avg = get_own_price_avg(df)
    comp_avg, comp_stats = get_comp_stats(df)

    # 병합
    result = own_avg.merge(comp_stats, on=KEYS, how='left')

    # ── 축1: SRP vs 경쟁사 ──
    result['srp_vs_comp_median_pct'] = (
        (result['target_srp'] - result['comp_buybox_median']) / result['comp_buybox_median']
    ).round(4)

    result['srp_vs_comp_bsr_wavg_pct'] = (
        (result['target_srp'] - result['comp_buybox_bsr_wavg']) / result['comp_buybox_bsr_wavg']
    ).round(4)

    # SRP percentile rank
    def srp_pct_rank(group):
        key = tuple(group[KEYS].iloc[0])
        srp = own_avg.set_index(KEYS).loc[key, 'target_srp']
        prices = group['comp_buybox_avg'].dropna()
        return round((prices < srp).mean(), 4) if len(prices) > 0 else np.nan

    pct = comp_avg.groupby(KEYS).apply(srp_pct_rank).reset_index(name='srp_percentile_rank')
    result = result.merge(pct, on=KEYS, how='left')

    # ── 축2: SRP vs Buybox (가격 통제력) ──
    result['srp_vs_buybox_gap'] = (result['target_srp'] - result['buybox_price']).round(2)
    result['srp_vs_buybox_pct'] = (
        result['srp_vs_buybox_gap'] / result['target_srp']
    ).round(4)

    # buybox도 경쟁사 대비
    result['buybox_vs_comp_median_pct'] = (
        (result['buybox_price'] - result['comp_buybox_median']) / result['comp_buybox_median']
    ).round(4)

    # ── 4사분면 판정 ──
    srp_high = result['srp_vs_comp_median_pct'] > COMP_HIGH_THRESHOLD
    srp_low = result['srp_vs_comp_median_pct'] < COMP_LOW_THRESHOLD
    srp_ok = ~srp_high & ~srp_low
    drift = abs(result['srp_vs_buybox_pct']) >= DRIFT_THRESHOLD
    no_drift = ~drift

    conditions = [
        srp_high & drift,      # SRP 고가 + buybox 이탈
        srp_high & no_drift,   # SRP 고가 + buybox 정상
        srp_low & drift,       # SRP 저가 + buybox 이탈
        srp_low & no_drift,    # SRP 저가 + buybox 정상
        srp_ok & drift,        # SRP 적정 + buybox 이탈
        srp_ok & no_drift,     # SRP 적정 + buybox 정상
    ]
    choices = [
        'SRP_HIGH_DRIFT',
        'SRP_TOO_HIGH',
        'SRP_LOW_DRIFT',
        'SRP_TOO_LOW',
        'BUYBOX_DRIFT',
        'ADEQUATE',
    ]
    result['srp_assessment'] = np.select(conditions, choices, default='CHECK')

    # buybox 이탈 방향 (양수=buybox<SRP, 음수=buybox>SRP)
    result['drift_direction'] = np.where(
        result['srp_vs_buybox_pct'] > 0, 'buybox_below_srp',
        np.where(result['srp_vs_buybox_pct'] < 0, 'buybox_above_srp', 'match')
    )

    # 권장 SRP 범위
    result['recommended_srp_low'] = (result['comp_buybox_median'] * (1 + COMP_LOW_THRESHOLD)).round(2)
    result['recommended_srp_high'] = (result['comp_buybox_median'] * (1 + COMP_HIGH_THRESHOLD)).round(2)
    result['srp_in_range'] = (
        (result['target_srp'] >= result['recommended_srp_low']) &
        (result['target_srp'] <= result['recommended_srp_high'])
    )

    # 정렬
    priority = {
        'SRP_HIGH_DRIFT': 0, 'SRP_TOO_HIGH': 1, 'SRP_LOW_DRIFT': 2,
        'SRP_TOO_LOW': 3, 'BUYBOX_DRIFT': 4, 'ADEQUATE': 5, 'CHECK': 6
    }
    result['_sort'] = result['srp_assessment'].map(priority)
    result = result.sort_values(['_sort', 'srp_vs_comp_median_pct'], ascending=[True, False])
    result = result.drop(columns='_sort')

    return result


# ============================================================
# Collection 서머리
# ============================================================
def collection_summary(result):
    summary = result.groupby('collection').agg(
        asin_count=('asin', 'nunique'),
        adequate_count=('srp_assessment', lambda x: (x == 'ADEQUATE').sum()),
        avg_target_srp=('target_srp', 'mean'),
        avg_buybox=('buybox_price', 'mean'),
        avg_srp_buybox_pct=('srp_vs_buybox_pct', 'mean'),
        avg_comp_median=('comp_buybox_median', 'mean'),
        avg_srp_comp_pct=('srp_vs_comp_median_pct', 'mean'),
        avg_net_ppm=('net_ppm', 'mean'),
    ).reset_index()

    summary['adequate_rate'] = (summary['adequate_count'] / summary['asin_count']).round(2)

    return summary.sort_values('adequate_rate')


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    # df = df_source[df_source['date'] >= date(2026, 1, 1)]
    df = df_source[pd.to_datetime(df_source['date']) >= '2026-01-01']
    
    result = evaluate_srp(df)
    summary = collection_summary(result)

    display_cols = [
        'collection', 'asin', 'size', 'inch',
        'target_srp', 'buybox_price',
        'srp_vs_buybox_pct', 'drift_direction',
        'comp_buybox_mean', 'comp_buybox_median',
        'srp_vs_comp_median_pct', 'srp_vs_comp_bsr_wavg_pct',
        'srp_percentile_rank', 'srp_assessment',
        'recommended_srp_low', 'recommended_srp_high', 'srp_in_range',
        'net_ppm',
     ]

    print("=== ASIN별 SRP 적정성 ===")
    print(result[display_cols].to_string(index=False))
    print("\n=== Collection 서머리 ===")
    print(summary.to_string(index=False))

    # result[display_cols].to_csv('srp_evaluation_v3.csv', index=False)
    # summary.to_csv('srp_summary_v3.csv', index=False)

    #print("분석 코드 준비 완료.")

C:\Users\최태욱\AppData\Local\Temp\ipykernel_36940\3422936156.py:94: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bsr = comp_avg.groupby(KEYS).apply(bsr_wavg).reset_index(name='comp_buybox_bsr_wavg')


=== ASIN별 SRP 적정성 ===
collection       asin        size  inch  target_srp  buybox_price  srp_vs_buybox_pct  drift_direction  comp_buybox_mean  comp_buybox_median  srp_vs_comp_median_pct  srp_vs_comp_bsr_wavg_pct  srp_percentile_rank srp_assessment  recommended_srp_low  recommended_srp_high  srp_in_range  net_ppm
       FMS B0CSJK3863 Short Queen   8.0      199.99        176.51             0.1174 buybox_below_srp            158.00              158.00                  0.2658                    0.2658                  1.0 SRP_HIGH_DRIFT               134.30                181.70         False    51.63
   OLB FGM B0CKYZ6K4V        Twin   8.0      119.99        105.07             0.1243 buybox_below_srp             95.62               95.62                  0.2549                    0.2549                  1.0 SRP_HIGH_DRIFT                81.28                109.96         False    49.22
  OLB GTFM B0CKYZPPMJ        Twin   8.0      119.99        107.18             0.1068 buybox_below_srp 

C:\Users\최태욱\AppData\Local\Temp\ipykernel_36940\3422936156.py:128: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pct = comp_avg.groupby(KEYS).apply(srp_pct_rank).reset_index(name='srp_percentile_rank')
